# Introducción a la Ciencia de Datos: Tarea 1

Este notebook contiene el código de base para realizar la Tarea 1 del curso. Puede copiarlo en su propio repositorio y trabajar sobre el mismo.
Las **instrucciones para ejecutar el notebook** están en la [página inicial del repositorio](https://gitlab.fing.edu.uy/maestria-cdaa/intro-cd).

Se utiliza el lenguaje Python y la librería Pandas. Si no tiene ninguna familiaridad con la librería, se recomienda realizar algún tutorial introductorio (ver debajo).
También se espera que los alumnos sean proactivos a la hora de consultar las documentaciones de las librerías y del lenguaje, para entender el código provisto.
Además de los recursos provistos en la [página del curso](https://eva.fing.edu.uy/course/view.php?id=1378&section=1), los siguientes recursos le pueden resultar interesantes:
 - [Pandas getting started](https://pandas.pydata.org/docs/getting_started/index.html#getting-started) y [10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html): Son parte de la documentación en la página oficial de Pandas.
 - [Kaggle Learn](https://www.kaggle.com/learn): Incluye tutoriales de Python y Pandas.


Si desea utilizar el lenguaje R y está dispuesto a no utilizar (o traducir) este código de base, también puede hacerlo.

En cualquier caso, **se espera que no sea necesario revisar el código para corregir la tarea**, ya que todos los resultados y análisis relevantes deberían estar en el **informe en formato PDF**.

## Cargar bibliotecas (dependencias)
Recuerde instalar los requerimientos (`requirements.txt`) en el mismo entorno donde está ejecutando este notebook (ver [README](https://github.com/DonBraulio/introCD)).

In [ ]:
from time import time
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from datasets import load_dataset

# Agregue aqui el resto de las librerias que necesite
# from ...
# import ...

## Descarga del dataset
En esta tarea se utilizará una base de datos abierta que contiene artículos de noticias publicados en distintos medios de prensa, con la finalidad de realizar una clasificación de textos según el medio de prensa al que pertenecen. [Link](https://huggingface.co/datasets/rjac/all-the-news-2-1-Component-one?utm_source=chatgpt.com) \
\
Ejecute la siguiente celda para descargar los datos y cargarlos en un dataframe de pandas. La constante `DATA_PATH` determina la ubicación donde se almacenaran los datos. \
\
El dataset entero pesa ~8.3gb. Para evitar demoras en la descarga/procesamiento vamos a utilizar el parámetro `streaming=True` y hacer un muestreo aleatorio para descargar una porción de los datos lo más representativa posible.

In [ ]:
ds = load_dataset("tomas-gr/all-the-news-2-1-Component-one-sampled", split="train",cache_dir="../data", streaming=True)
df = ds.to_pandas()

## Lectura de Datos

In [ ]:
# Veamos las primeras filas del DataFrame
df.head()

In [ ]:
# Veamos información general del DataFrame
df.info()

# Parte 1: Cargado y Limpieza de Datos

## A. Exploración de Datos
Analice el contenido del DataFrame. Reporte si existen datos faltantes en algún campo, o cualquier otro problema de calidad de datos que encuentre. \
En particular, analice la cantidad de artículos por medio de prensa, y a partir de este punto trabaje con los **cinco medios con mayor cantidad de artículos**.

In [ ]:
# %pip install --upgrade setuptools ydata-profiling
%pip install "setuptools<82"

In [ ]:
from data_profiling import ProfileReport
profile = ProfileReport(
    df, 
    explorative=True
)

#profile.to_notebook_iframe()


In [ ]:
profile.to_file("publications_data_profiling_report.html")

In [ ]:
mask = df["url"].isnull() | df["publication"].isnull()
df[mask][["url", "publication", "author", "article"]].isnull().sum()

In [ ]:
df["publication"].nunique()

# TODO: Analice datos faltantes por columna
Total datos: 30213 filas

Columnas con datos faltantes:
- author: 18808
- article: 29037
- url: 30072
- section: 19981
- publication: 30072

Analisis: 
- Falta publication siempre que falta url (chequear si faltan todos los demas datos cuando faltan estos)
- Chequear si cuando falta article / section / author falta algun otro de los faltantes
html: Missing values - Matrix del data profiling

In [ ]:
# TODO: Analice la cantidad de artículos por medio de prensa

# Tome los 5 medios con más artículos
top_5_publications = (
    # Agrupar el DataFrame por la columna publication y contar la cantidad de articulos por cada medio
    df.groupby("publication")["article"].count()
    # ordenar de mayor a menor y tomar los 5 primeros, y quedarnos solo con el nombre del medio (index)
    .sort_values(ascending=False).head(5).index
)
# Filtrar el dataframe para incluir solo las filas en donde la columna publication esta dentro de top_5_publications
df_top_5 = df[df["publication"].isin(top_5_publications)].copy()
#df_top_5 # 18771 filas

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

pub_counts = df_top_5["publication"].value_counts().reset_index()
pub_counts.columns = ["publication", "n_articles"]
pub_counts = pub_counts.sort_values("n_articles", ascending=False)

colors = sns.color_palette("viridis", n_colors=len(pub_counts))

ax.bar(pub_counts["publication"], pub_counts["n_articles"], color=colors, width=0.6)

ax.set_title("Distribución de artículos por medio de prensa (top 5)", fontsize=18, pad=20)
ax.set_xlabel("Medio de prensa", fontsize=14)
ax.set_ylabel("Número de artículos", fontsize=14)
ax.tick_params(axis="both", labelsize=14)
ax.set_xticklabels(pub_counts["publication"].str.replace("The New York Times", "NYT"), fontsize=15)

sns.despine()
plt.tight_layout()
plt.savefig("../Tarea_1/figs/fig_distribucion_pubs.png", dpi=150, bbox_inches="tight")
plt.show()

## B. Visualización temporal
Genere una gráfica que permita visualizar los artículos de los cinco medios a lo largo del tiempo, con alguna escala temporal adecuada. \
Comente si se identifican momentos de mayor actividad o patrones temporales en la cobertura.

In [ ]:
# TODO: Visualización de los artículos de cada medio a lo largo del tiempo
# Preste especial atención al formato de la columna 'date', ya que puede contener diferentes formatos de fecha.

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")

df_top_5["datetime"] = pd.to_datetime(df_top_5["date"], format='mixed', errors='raise')

articles_per_month = (
    df_top_5
    .groupby(["publication", pd.Grouper(key="datetime", freq="ME")])
    .size()
    .reset_index(name="n_articles")
)
articles_per_month["publication"] = articles_per_month["publication"].replace(
    "The New York Times", "NY Times"
)

fig, ax = plt.subplots(figsize=(16, 8))

viridis_colors = sns.color_palette("viridis", 5)

palette = {
    "Reuters": viridis_colors[0],      # dark purple
    "The Hill": viridis_colors[1],     # blue
    "CNBC": viridis_colors[2],         # teal
    "People": viridis_colors[3],       # green
    "NY Times": viridis_colors[4],     # yellow
}
sns.lineplot(
    data=articles_per_month,
    x="datetime",
    y="n_articles",
    hue="publication",
    palette="viridis",
    linewidth=3.5,
    markers=True,
    markersize=7,
    ax=ax
)

ax.grid(True, which="major", linestyle="--", linewidth=0.5, alpha=0.4)

ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

ax.set_title("Artículos publicados por mes (top 5 medios)", fontsize=24, pad=20)
ax.set_xlabel("Fecha", fontsize=18)
ax.set_ylabel("Artículos por mes", fontsize=18)
ax.tick_params(axis="both", labelsize=15)

ax.legend(
    title="Medio de prensa",
    title_fontsize=15,
    fontsize=14,
    loc="upper left",
    frameon=True,
    framealpha=0.85,
)

sns.despine()
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("../Tarea_1/figs/monthly_article_vol_news_source.png", dpi=150, bbox_inches="tight")
plt.show()

## C. Limpieza de texto y conteo de palabras
Se provee la función `clean_text(...)` que realiza parte de la normalización del texto. \
**Complete la función** agregando signos de puntuación faltantes y cualquier otra normalización que considere oportuna. \
Compruebe el resultado observando el contenido del DataFrame procesado. Comente todas las transformaciones que haya agregado y justifique.

In [ ]:
def clean_text(df: pd.DataFrame, column_name: str) -> pd.Series:
    """
    Clean a text column by:
    - removing content before the first newline (dateline header)
    - lowercasing
    - normalizing apostrophes to standard single quote
    - removing punctuation except for apostrophes
    - removing numeric tokens
    - collapsing repeated whitespace
    """

    result = (
        df[column_name]
        .fillna("")
        .astype(str)
        # remove everything before first newline (dateline)
        .str.replace(r"^[^\n]*\n", "", regex=True)
        # lowercase
        .str.lower()
        # normalize apostrophes to standard single quote
        .str.replace(r"[''`]", "'", regex=True)
        # remove punctuation, keeping alphanumeric, whitespace and apostrophes
        .str.replace(r"[^\w\s']", " ", regex=True)
        # remove numeric tokens
        .str.replace(r"\d+", " ", regex=True)
        # collapse repeated whitespace
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    return result


In [ ]:
# TODO: Aplique clean_text sobre la columna de texto elegida y cree una nueva columna "CleanText"

df_top_5["clean_text_article"] = clean_text(df_top_5, "article")
pd.set_option("display.max_colwidth", 150)

df_top_5[["article", "clean_text_article"]].head()

In [ ]:
df_top_5["clean_text_title"] = clean_text(df_top_5, "title")
df_top_5[["title", "clean_text_title"]].head()

## D. Elección de campos de texto
Discuta si conviene trabajar con:
- sólo el cuerpo del artículo,
- sólo el título,
- o una combinación de ambos.

Justifique brevemente su decisión.

*TODO: Escriba su análisis en el informe.*

## E. Pistas que identifican al medio de prensa
Analice si en el texto aparecen pistas que identifiquen de manera directa al medio de prensa (nombres del medio, URLs, firmas, nombres de secciones, plantillas repetidas, etc.). \
En caso de encontrarlas, comente si considera conveniente eliminarlas o reducir su impacto, y justifique su decisión.

In [ ]:
source_identifiers = {
    "Reuters": ["reuters"],
    "CNBC": ["cnbc"],
    "The Hill": ["the hill", "thehill"],
    "The New York Times": ["new york times", "nytimes", "nyt"],
    "People": ["people"],
}

source_domains = {
    "Reuters": r"reuters\.com",
    "CNBC": r"cnbc\.com",
    "The Hill": r"thehill\.com",
    "The New York Times": r"nytimes\.com",
    "People": r"people\.com",
}

rows = []

for source in ["Reuters", "The Hill", "CNBC", "The New York Times", "People"]:
    own = df_top_5[df_top_5["publication"] == source].copy()
    n_own = len(own)

    # ----------------------------
    # 1. Lexical mentions in ARTICLE (cleaned)
    # ----------------------------
    name_mask_article = pd.Series(False, index=own.index)
    for name in source_identifiers[source]:
        name_mask_article |= own["clean_text_article"].str.contains(
            name, case=False, na=False
        )

    # ----------------------------
    # 2. Lexical mentions in TITLE (cleaned)
    # ----------------------------
    name_mask_title = pd.Series(False, index=own.index)
    for name in source_identifiers[source]:
        name_mask_title |= own["clean_text_title"].str.contains(
            name, case=False, na=False
        )

    name_mask = name_mask_article | name_mask_title

    # ----------------------------
    # 3. URL/domain signal in RAW article text
    # ----------------------------
    url_mask = own["article"].fillna("").str.contains(
        source_domains[source], case=False, regex=True
    )

    # ----------------------------
    # Combine signals
    # ----------------------------
    any_mask = name_mask | url_mask
    n_match = any_mask.sum()

    rows.append({
        "Publicación": source,
        "Total artículos": n_own,
        "Con auto-mención": n_match,
        "Porcentaje (%)": round(100 * n_match / n_own, 1),
    })

pd.DataFrame(rows)

## F. Restricción por sección o período temporal
Evalúe si conviene restringir el análisis a artículos de una misma sección temática o de un período temporal acotado, con el objetivo de reducir el efecto del tema sobre una futura tarea de clasificación por medio. \
No es necesario implementar esta restricción, pero sí discutir sus posibles ventajas y desventajas.

In [ ]:
for pub in sorted(df_top_5["publication"].unique()):
    sections = df_top_5[df_top_5["publication"] == pub]["section"].dropna().unique()
    print(f"\n{pub}:")
    print(sorted(sections))

In [ ]:
coverage = (
      df_top_5.groupby("publication")["datetime"]
      .agg(first="min", last="max")
      .sort_values("last")
  )

common_start = coverage["first"].max()
common_end   = coverage["last"].min()
print(f"Common window (all 5 sources active): {common_start.date()}  →  {common_end.date()}")


In [ ]:
df["datetime"] = pd.to_datetime(df["date"], format="mixed", errors="raise")
coverage = (
      df.groupby("publication")["datetime"]
      .agg(first="min", last="max")
      .sort_values("last")
  )

common_start = coverage["first"].max()
common_end   = coverage["last"].min()
print(f"Common window (all sources active): {common_start.date()}  →  {common_end.date()}")
print(coverage.to_string())

In [ ]:
top_5_mask = (df["datetime"] >= "2016-01-01") & (df["datetime"] <= "2019-12-01")
mask = (df["datetime"] >= "2017-05-01") & (df["datetime"] <= "2018-08-31")
#df[mask].groupby("publication")["article"].count().sort_values(ascending=False)
print("Top 5 - coverage:", df[top_5_mask]["article"].count())
print("All - coverage:", df[mask]["article"].count())

*TODO: Escriba su análisis en el informe.*

# Parte 2: Conteo de Palabras y Visualizaciones

## A. Palabras más frecuentes por medio
Realice una visualización que permita comparar las palabras más frecuentes de cada uno de los cinco medios de prensa. \
Sin necesidad de implementarlo, proponga ideas para modificar esta visualización con el fin de encontrar diferencias entre secciones temáticas, fechas, o tipos de noticias.

In [ ]:
# TODO: Realice una visualización que permita comparar las palabras más frecuentes
# de cada uno de los cinco medios de prensa.
# - ¿Encuentra algún problema en los resultados?
from wordcloud import STOPWORDS, WordCloud
import matplotlib.pyplot as plt
print(STOPWORDS)


In [ ]:
# Si no se incluye lista de stopwords, las palabras mas frecuentes son "a", "to", "in", "of", "for"
# Por defecto, word cloud incluye la lista basica de stopwords
def plot_wordcloud(
    df,
    columns,
    title="Word Cloud",
    figsize=(14, 7),
    colormap="viridis",
    max_words=150,
    background_color="white"
):
    """
    Generate a word cloud from pre-cleaned text columns.

    Parameters:
    - df: pandas DataFrame (assumes text already cleaned)
    - columns: str or list of str
    - title: plot title
    """

    if isinstance(columns, str):
        columns = [columns]

    # combine text efficiently
    text = df[columns].fillna("").astype(str).agg(" ".join, axis=1)
    text = " ".join(text.tolist())

    wordcloud = WordCloud(
        width=1200,
        height=600,
        background_color=background_color,
        colormap=colormap,
        max_words=max_words,
        collocations=False  # avoids artificial bigrams
    ).generate(text)

    plt.figure(figsize=figsize)
    plt.imshow(wordcloud, interpolation="bilinear")
    plt.axis("off")
    plt.title(title, fontsize=16, pad=15)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_wordcloud(
    df_top_5,
    columns="clean_text_title",
    title="Headlines Word Cloud (Top 5 Media Sources)"
)

In [ ]:
plot_wordcloud(
    df_top_5,
    columns="clean_text_article",
    title="Article Text Word Cloud (Top 5 Media Sources)"
)

In [ ]:
def wordcloud_by_source(
    df,
    text_column,
    source_column="publication",
    max_words=150,
    colormap="viridis",
    save_path=None,
    stopwords=STOPWORDS
):
    """
    Generate one word cloud per news source using a shared colormap.
    """

    sources = df[source_column].dropna().unique()
    n_sources = len(sources)

    # ----------------------------
    # Grid configuration (2 columns)
    # ----------------------------
    cols = 2
    rows = (n_sources + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols)
    axes = axes.flatten()

    # ----------------------------
    # Wordcloud per source
    # ----------------------------
    for i, source in enumerate(sources):
        subset = df[df[source_column] == source]
        text = " ".join(subset[text_column].astype(str).tolist())

        wc = WordCloud(
            width=800,
            height=400,
            background_color="white",
            colormap=colormap,
            max_words=max_words,
            collocations=False,
            stopwords=stopwords
        ).generate(text)

        axes[i].imshow(wc, interpolation="bilinear")
        axes[i].set_title(source, fontsize=12)
        axes[i].axis("off")

    # ----------------------------
    # Turn off unused subplots
    # ----------------------------
    for j in range(n_sources, len(axes)):
        axes[j].axis("off")

    fig.subplots_adjust(hspace=0.2, wspace=0.1)
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")

    plt.show()

In [ ]:
wordcloud_by_source(
    df_top_5,
    "clean_text_title",
    save_path="../Tarea_1/figs/wordcloud_title.png",
)


In [ ]:
wordcloud_by_source(
    df_top_5,
    "clean_text_article",
    save_path="../Tarea_1/figs/wordcloud_article.png",
    stopwords=STOPWORDS.union(
        {
            "said", "says", "will", "new", "one", "people", "also", "us", "trump", "year", "mr", 
            "hill", "reuters", "cnbc", "thehill", "thehill", "new york times", "nytimes", "nyt", "people", "t", "u", "s", "ms", "mrs"
        }
    )
)


## B. Medios con mayor cantidad de palabras
Corra el código que permite encontrar los medios con mayor cantidad de palabras. \
En caso de encontrar algún problema luego de realizar la visualización, comente a qué se debe y proponga formas de resolverlo.

In [ ]:
# TODO: Busque los medios con mayor cantidad de palabras
word_counts = (
    df_top_5
    .assign(word_count=df_top_5["clean_text_article"].fillna("").str.split().str.len())
    .groupby("publication")["word_count"]
    .sum()
    .sort_values(ascending=False)
)

In [ ]:
word_counts

In [ ]:
# TODO: Busque los medios con mayor cantidad de palabras
df_top_5["word_count"] = df_top_5["clean_text_article"].fillna("").str.split().str.len()
word_stats = (
      df_top_5.groupby("publication")["word_count"]
      .agg(total="sum", media="mean", mediana="median")
      .round(1)
      .sort_values("total", ascending=False)
  )
word_stats

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mticker

word_counts.index = word_counts.index.str.replace("The New York Times", "NY Times")

fig, ax = plt.subplots(figsize=(9, 6))

sns.barplot(
    x=word_counts.index,
    y=word_counts.values,
    hue=word_counts.index,
    palette="viridis",
    legend=False,
    width=0.6,
    ax=ax,
)

ax.yaxis.set_major_formatter(mticker.EngFormatter())
ax.set_title("Total de palabras por medio de prensa", fontsize=18, pad=15)
ax.set_xlabel("")
ax.set_ylabel("Total de palabras", fontsize=14)
ax.tick_params(axis="both", labelsize=14)
ax.grid(True, axis="y", linestyle="--", linewidth=0.7, alpha=0.8)

sns.despine()
plt.tight_layout()
plt.savefig("../Tarea_1/figs/total_words_news_source.png", dpi=150, bbox_inches="tight")
plt.show()  

Esto podria generar un problema de interpretacion, lo unico que nos dice esto es que medio tiene la mayor cantidad de palabras en total. Se podria hacer un analisis promediando palabras por articulo.

In [ ]:
df_top_5 = df_top_5.copy()

df_top_5["word_count"] = df_top_5["clean_text_article"].fillna("").str.split().str.len()

avg_words = (
    df_top_5
    .groupby("publication")["word_count"]
    .mean()
    .sort_values(ascending=False)
)

In [ ]:
order = ["NY Times", "Reuters", "The Hill", "CNBC", "People"]
df_top_5["publication_label"] = df_top_5["publication"].replace("The New York Times", "NY Times")
fig, ax = plt.subplots(figsize=(9, 6))

sns.boxplot(
    data=df_top_5,
    x="publication_label",
    y="word_count",
    order=order,
    palette="viridis",
    showfliers=False,
    width=0.5,
    linewidth=1.5,
    ax=ax,
)

ax.set_title("Longitud de artículos por medio de prensa", fontsize=18, pad=15)
ax.set_xlabel("")
ax.set_ylabel("Palabras por artículo", fontsize=14)
ax.tick_params(axis="both", labelsize=14)
ax.grid(True, axis="y", linestyle="--", linewidth=0.7, alpha=0.8)

sns.despine()
plt.tight_layout()
plt.savefig("../Tarea_1/figs/article_length_distrib_news_source.png", dpi=150, bbox_inches="tight")
plt.show()

## C. Matriz de menciones entre medios
Construya una matriz de 5×5, donde cada fila y columna corresponden a un medio de prensa, y la entrada (i,j) contiene la cantidad de veces que el medio *i* menciona al medio *j*. \
\
**Opcional:** genere un grafo dirigido con esa matriz de adyacencia para visualizar las menciones. Puede ser útil la biblioteca `networkx`.

In [ ]:
# TODO: Construya una matriz de 5x5, donde cada fila y columna corresponden a un medio de prensa,
# y la entrada (i,j) contiene la cantidad de veces que el medio "i" menciona al medio "j".

sources = df_top_5["publication"].unique()
mentions_matrix = pd.DataFrame(0, index=sources, columns=sources)
for _, row in df_top_5.iterrows():
    source_i = row["publication"] # media source of the current article
    text = str(row["clean_text_article"]) # article text to search for mentions

    for source_j in sources:
        # avoid self-mentions unless desired
        if source_i == source_j:
            continue

        if source_j.lower() in text:
            mentions_matrix.loc[source_i, source_j] += 1

In [ ]:
plt.figure(figsize=(8, 6))

sns.heatmap(
    mentions_matrix,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.title("Media Mention Matrix")
plt.xlabel("Mentioned Source")
plt.ylabel("Source")
plt.show()

In [ ]:
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import numpy as np

G = nx.from_pandas_adjacency(mentions_matrix, create_using=nx.DiGraph)
G.remove_edges_from(
    [(u, v) for u, v, d in G.edges(data=True) if d["weight"] == 0]
)

plt.figure(figsize=(12, 10))
pos = nx.spring_layout(G, seed=42)
ax = plt.gca()

# 1. Generate a unique color for each distinct node using the Viridis palette
nodes = list(G.nodes())
num_nodes = len(nodes)

palette = cm.viridis(np.linspace(0, 1, num_nodes))
node_color_map = {node: palette[i] for i, node in enumerate(nodes)}


# 2. Draw nodes using their assigned source colors
for node in G.nodes():
    nx.draw_networkx_nodes(
        G,
        pos,
        nodelist=[node],
        node_size=2400,
        node_color=[node_color_map[node]],
    )

# 3. Add text labels inside the nodes
nx.draw_networkx_labels(G, pos, font_size=10, font_weight="bold", font_color="lightblue")

# 4. Draw edges and text weights using the source node's color
for u, v, data in G.edges(data=True):
    start = pos[u]
    end = pos[v]
    # The line and arrow inherit the exact color of the source node (u)
    edge_color = node_color_map[u]

    arrow = plt.matplotlib.patches.FancyArrowPatch(
        posA=start,
        posB=end,
        connectionstyle="arc3,rad=0.25",  # Keeps overlapping lines separated
        arrowstyle="-|>",
        mutation_scale=20,
        color=edge_color,
        linewidth=2.5,  # Thicker lines make the source connection obvious
        shrinkA=24,
        shrinkB=24,
    )
    ax.add_patch(arrow)

    # Calculate exact midpoint of the curved line for the number bubble
    mid_x = (start[0] + end[0]) / 2 + (end[1] - start[1]) * 0.25 * 0.5
    mid_y = (start[1] + end[1]) / 2 - (end[0] - start[0]) * 0.25 * 0.5

    # White bubble background with a border matching the source color
    plt.text(
        mid_x,
        mid_y,
        str(data["weight"]),
        color="black",
        fontsize=10,
        fontweight="bold",
        ha="center",
        va="center",
        bbox=dict(
            boxstyle="round,pad=0.2",
            facecolor="white",
            edgecolor=edge_color,
            linewidth=1.5,
        ),
    )

plt.title("Media Mention Graph", fontsize=14, pad=20)
plt.axis("off")
plt.tight_layout()
plt.show()


## D. Preguntas propuestas
Proponga al menos tres preguntas que se podrían intentar responder a partir de estos datos, y mencione posibles caminos para responderlas, sin implementar nada.

*TODO: Escriba sus preguntas y posibles caminos en el informe.*
